<a href="https://colab.research.google.com/github/muntakson/omics-atlas-workshop/blob/main/notebooks/02_differential_expression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧬 Workshop 2 — Differential expression with PyDESeq2

**Companion to [Chapter 2: RNA-Seq Data Analysis — A Practical Guide](https://omics.iotok.org/chapters/rnaseq-practical-guide)**.

You will run the **DESeq2** workflow (the Python port, PyDESeq2) to find genes that change between dexamethasone-treated and control samples, then reproduce the classic **volcano**, **MA** and **heatmap** figures from Chapter 2.

## 1. Install & load
`pydeseq2` and `mygene` aren't in the Colab base image, so we install them (≈30 s).

In [ ]:
!pip install -q pydeseq2 mygene

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
sns.set_style('whitegrid')
RAW = 'https://raw.githubusercontent.com/muntakson/omics-atlas-workshop/main'
counts = pd.read_csv(f'{RAW}/data/airway_scaledcounts.csv', index_col=0)
meta   = pd.read_csv(f'{RAW}/data/airway_metadata.csv', index_col=0)

## 2. Prepare the matrix
PyDESeq2 wants **samples × genes**, integer counts. We transpose, round, and drop genes with almost no reads.

In [ ]:
counts_t = counts.T.round().astype(int)
counts_t = counts_t.loc[:, counts_t.sum(axis=0) >= 10]   # filter low-count genes
print('samples x genes:', counts_t.shape)

## 3. Run DESeq2
The model `~dex` asks: *for each gene, does expression differ between the `dex` groups?*

In [ ]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

dds = DeseqDataSet(counts=counts_t, metadata=meta, design='~dex')
dds.deseq2()

ds = DeseqStats(dds, contrast=['dex', 'treated', 'control'])
ds.summary()
res = ds.results_df
res.head()

## 4. Map Ensembl IDs → gene symbols
Results are keyed by Ensembl gene IDs. We attach human-readable symbols with `mygene` (needed for enrichment in Workshop 3).

In [ ]:
import mygene
mg = mygene.MyGeneInfo()
q = mg.querymany(list(res.index), scopes='ensembl.gene', fields='symbol',
                 species='human', as_dataframe=True, verbose=False)
res['symbol'] = res.index.map(q['symbol'].groupby(level=0).first())
res.index.name = 'ensgene'
res.dropna(subset=['padj']).sort_values('padj').head()

## 5. How many genes are significant?
A common cutoff: adjusted p-value (**padj**) < 0.05 **and** |log2 fold-change| > 1 (a 2-fold change).

In [ ]:
sig  = res[(res.padj < 0.05) & (res.log2FoldChange.abs() > 1)].dropna(subset=['symbol'])
up   = sig[sig.log2FoldChange > 0]
down = sig[sig.log2FoldChange < 0]
print(f'{len(sig)} DEGs  |  {len(up)} up  |  {len(down)} down')
up.sort_values('log2FoldChange', ascending=False)[['symbol','log2FoldChange','padj']].head(10)

## 6. Volcano plot
Each dot is a gene: fold-change (x) vs significance (y). The named genes at top-right are the strongest dexamethasone responders.

In [ ]:
r = res.dropna(subset=['padj']).copy()
r['sig'] = (r.padj < 0.05) & (r.log2FoldChange.abs() > 1)
plt.figure(figsize=(6,5))
plt.scatter(r.log2FoldChange, -np.log10(r.padj), s=6, c=r['sig'].map({True:'#d64545',False:'#b0b7c6'}), alpha=.5)
for _, g in up.sort_values('log2FoldChange', ascending=False).head(6).iterrows():
    plt.annotate(g['symbol'], (g['log2FoldChange'], -np.log10(g['padj'])), fontsize=8)
plt.axhline(-np.log10(.05), ls='--', c='grey'); plt.axvline(1, ls='--', c='grey'); plt.axvline(-1, ls='--', c='grey')
plt.xlabel('log2 fold change (treated / control)'); plt.ylabel('-log10 adjusted p'); plt.title('Volcano plot')
plt.tight_layout(); plt.show()

## 7. Heatmap of top DEGs
Reproduces the Chapter 2 heatmap: rows = top genes, columns = samples, clustered.

In [ ]:
import numpy as np
cpm = np.log2(counts / counts.sum(axis=0) * 1e6 + 1)
topg = sig.reindex(sig.log2FoldChange.abs().sort_values(ascending=False).index).head(25)
mat = cpm.loc[topg.index]
mat.index = topg['symbol'].values
col_colors = meta['dex'].map({'control':'#2f6fed','treated':'#12a594'})
sns.clustermap(mat, z_score=0, cmap='RdBu_r', col_colors=col_colors, figsize=(6,8))
plt.show()

### ✏️ Your turn
CRISPLD2 is the gene the original paper highlighted. Find its row in `res` (symbol == 'CRISPLD2') — is it significantly changed?

In [ ]:
# your code here


<details><summary>▶ Solution</summary>

In [ ]:
res[res['symbol'] == 'CRISPLD2'][['symbol','log2FoldChange','padj']]

</details>

## ✅ Recap
You ran a full DESeq2 analysis on real data and found the differentially expressed genes.

**Next → [Workshop 3: Pathway enrichment](https://colab.research.google.com/github/muntakson/omics-atlas-workshop/blob/main/notebooks/03_pathway_enrichment.ipynb)** — what do these genes *do* together?